# ReadtheDocs Retrieval Augmented Generation (RAG) using Zilliz Free Tier

在本笔记本中，我们将使用 Milvus 文档页面来创建一个关于我们产品的聊天机器人。该聊天机器人将遵循 RAG（检索增强生成）流程：通过语义向量搜索获取数据片段，然后将问题与上下文作为提示输入大语言模型，以生成回答。

许多 RAG 演示使用 OpenAI 的嵌入模型和 ChatGPT 生成式 AI 模型。**而在本笔记本中，我们将演示一个完全开源的 RAG 系统栈。**

使用开源的问答系统并结合检索可以节省成本，因为我们几乎每次都能免费调用自有数据进行检索、评估和开发迭代。仅在最终生成对话时，才向 OpenAI 发起一次付费调用。

<img src="./rag_image.png">

Let's get started!

In [1]:
# For colab install these libraries in this order:
# !python -m pip install torch transformers sentence-transformers langchain
# !python -m pip install -U pymilvus 'pymilvus[model]'
# !python -m pip install unstructured openai tqdm numpy ipykernel

In [2]:
import sys,os,time,pprint
import numpy as np

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'

os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

os.environ['TORCH_HOME'] = CUSTOM_CACHE

## Download Milvus documentation

我们将使用的数据是自己产品文档的网页。ReadTheDocs 是一个开源免费的软件文档托管平台，其文档使用 Sphinx 文档生成器编写。

下面的代码块会将网页下载到名为 `rtdocs` 的本地目录中。

我已经将 `rtdocs` 数据文件夹上传到了 GitHub，因此如果你克隆了我的仓库，应该能看到它。

### UNCOMMENT TO DOWNLOAD THE DOCS.

In [3]:
# # # !pip install -U langchain
# from langchain_community.document_loaders import RecursiveUrlLoader
#
# DOCS_PAGE="https://milvus.io/docs/"
#
# loader = RecursiveUrlLoader(DOCS_PAGE)
# docs = loader.load()
#
# num_documents = len(docs)
# print(f"loaded {num_documents} documents")

### READ THE DOCS FROM A LOCAL DIRECTORY

In [4]:
from langchain_community.document_loaders import DirectoryLoader

# Load HTML files from a local directory
path='./rtdocs_new'
loader=DirectoryLoader(path,glob='*.html')
docs=loader.load()

num_documents=len(docs)
print(f"loaded {num_documents} documents")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_17248\3423029276.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


loaded 22 documents


## Start up Milvus running in local Docker

> ⛔️ 请确保正确安装 pymilvus 的版本以及 server yml 文件。所有版本（主版本和次版本）必须完全匹配。

1. 安装 Docker
2. 启动您的 Docker Desktop
3. 下载最新的 docker-compose.yml 文件（或运行 wget 命令，将 version 替换为您当前使用的版本）
> wget https://github.com/milvus-io/milvus/releases/download/v2.4.0-rc.1/milvus-standalone-docker-compose.yml -O docker-compose.yml

4. 在终端中：
    - 进入保存 .yml 文件的目录（通常与本笔记本在同一目录下）
    - docker compose up -d
    - 通过终端或 Docker Desktop 验证容器是否正在运行

5. 从代码中（参见下方笔记本代码）：
    - 导入 milvus
    - 连接到本地 milvus 服务器

### STEP 1. CONNECT TO MILVUS STANDALONE DOCKER.

In [5]:
import pymilvus
from pymilvus import MilvusClient

print(f"Pymilvus: {pymilvus.__version__}")

# 1. 创建客户端（自动连接）
mc = MilvusClient(uri="http://localhost:19530")

# 2. 获取服务器版本（使用客户端方法）
print(mc.get_server_version())


Pymilvus: 3.0.1
3.0.0


## Load the Embedding Model checkpoint and use it to create vector embeddings

#### What are Embeddings?

请查看此博客，了解嵌入的入门知识。

一个很好的起点是从 HuggingFace MTEB 领先榜单中选择嵌入模型，按“检索平均值”列降序排列，因为该任务与 RAG 最为相关。然后选择排名最低、但性能最高的嵌入模型。但请注意！部分列出的模型可能过度拟合训练数据，因此无法在您的数据上达到承诺的性能表现。

Milvus（以及 Zilliz）仅支持经过验证且未过度拟合的嵌入模型。

在本笔记中，我们将使用开源的 **BGE-M3 模型**，它支持以下功能：
- 超过 100 种语言
- 上下文长度最长可达 8192
- 多种嵌入推理方式，包括密集型（语义）、稀疏型（词汇）以及多向量 Colbert 重排序

BGE-M3 是首个同时支持三种检索方法的嵌入模型，已在多语言（MIRACL）和跨语言（MKQA）基准测试中取得前沿性能。论文及 HuggingFace 信息详见此处。

Milvus 是全球首个开源向量数据库，在生成式 AI 工作流中发挥着关键作用，提供可扩展、高效的存储与搜索能力，助力语义搜索。其高级功能包括元数据过滤和混合搜索。自2.4版本起，Milvus已内置对BGE M3的支持。

<img src="./bge_m3.png">

### STEP 2. DOWNLOAD AN OPEN SOURCE EMBEDDING MODEL.

In [6]:
from pymilvus.model.hybrid import BGEM3EmbeddingFunction
import torch

# Initialize torch settings
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"DEVICE: {DEVICE}")

# Initialize a Milvus build-in spare-dense-reranking encoder
embedding_model=BGEM3EmbeddingFunction(
    device=str(DEVICE),
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
)
EMBEDDING_DIM=embedding_model.dim['dense']

print(f"dense_dim: {EMBEDDING_DIM}")
print(f"sparse_dim: {embedding_model.dim['sparse']}")
print(f"colbert_dim: {embedding_model.dim['colbert_vecs']}")

DEVICE: cuda


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

dense_dim: 1024
sparse_dim: 250002
colbert_dim: 1024


## Create a Milvus collection

在 Milvus 中，你可以将集合类比为 SQL 数据库中的“表”。该集合将包含以下内容：

- **模型架构**（或无架构的 Milvus 客户端）

    💡 你需要从嵌入模型中获取向量的 EMBEDDING_DIM 参数。常见取值如下：
    - sbert 嵌入模型：1024
    - ada-002 OpenAI 嵌入模型：1536

- **向量索引**，用于高效向量搜索
- **向量距离度量**，用于计算最近邻向量
- **一致性级别**：Milvus 支持事务一致性，但根据 CAP 定理，必须牺牲一定的延迟。💡 由于电影评论搜索并非关键任务，因此`eventually`在此处是可接受的。

## Add a Vector Index

向量索引用于确定在用户提交查询时，查找数据中与该查询最接近的向量所采用的向量搜索算法。

大多数向量索引根据数据库的使用场景（插入向量或搜索向量）而采用不同的参数组合：

- **插入向量**（创建模式）
- **搜索向量**（搜索模式）

请向下滚动文档页面，查看 Milvus 提供的不同向量索引列表。例如：

- FLAT — 确定性穷尽搜索
- IVF_FLAT 或 IVF_SQ8 — 哈希索引（随机近似搜索）
- HNSW — 图形索引（随机近似搜索）
- AUTOINDEX — 根据 OSS 与 Zilliz 云、GPU 类型及数据规模自动确定

除了搜索算法外，我们还需要指定**距离度量**，即定义向量空间中“接近”的标准。在下方单元格中选择了 `HNSW` 搜索索引。其可用的距离度量包括：

- L2 — L2 范数
- IP — 点积
- COSINE — 角度距离

💡 大多数应用场景更适合使用归一化嵌入（normalized embeddings），此时 L2 不适用（每个向量长度为1），IP 和 COSINE 相等。仅当您计划保持嵌入未归一化时，才应选择 L2。

### STEP 3. CREATE A NO-SCHEMA MILVUS COLLECTION AND DEFINE THE DATABASE INDEX.

In [7]:
from pymilvus import DataType

# Set the Milvus collection name.
COLLECTION_NAME = "MilvusDocs"

# Specify the data schema for the new Collection.
MAX_LENGTH=65535

# pymilvus 2.6+ 的 create_collection 不再接受 fields=[...] 列表(那是旧版快速建表,
# 只支持单个 dense 向量字段,并强制要求 dimension 参数)。
# 要同时存 sparse + dense,必须用显式 schema 建表:
#   - dense_vector  写 dim=EMBEDDING_DIM(1024)
#   - sparse_vector 不写 dim(稀疏向量变长,维度由数据决定,即词表大小 250002)
schema = mc.create_schema(auto_id=True, enable_dynamic_field=True)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, auto_id=True)
schema.add_field(field_name="sparse_vector", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="dense_vector", datatype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM)
schema.add_field(field_name="chunk", datatype=DataType.VARCHAR, max_length=MAX_LENGTH)
schema.add_field(field_name="source", datatype=DataType.VARCHAR, max_length=MAX_LENGTH)
schema.add_field(field_name="h1", datatype=DataType.VARCHAR, max_length=100)
schema.add_field(field_name="h2", datatype=DataType.VARCHAR, max_length=MAX_LENGTH)

# Check if collection already exists, if so drop it.
has=mc.has_collection(COLLECTION_NAME)
if has:
    drop_result=mc.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection {COLLECTION_NAME}")

# Sparse index: IP metric, SPARSE_INVERTED_INDEX.
sparse_index={
    "index_type":"SPARSE_INVERTED_INDEX",
    "metric_type":"IP"
}
# Dense index: HNSW + COSINE.
    # M = max number graph connections per layer. Large M = denser graph.
    # Choice of M: 4~64, larger M for larger data and larger embedding lengths.
M = 16
# efConstruction = num_candidate_nearest_neighbors per layer.
    # Use Rule of thumb: int. 8~512, efConstruction = M * 2.
efConstruction = M * 2
dense_index={
    "index_type":"HNSW",
    "metric_type":"COSINE",
    "params":{"M":M,"efConstruction":efConstruction}
}

# 2.6+ 的 create_index 需要 IndexParams 对象(用 prepare_index_params 构建),不能传 dict。
# 这里直接在 create_collection 时一并传入 schema + index_params,建表后自动建索引并加载。
index_params = mc.prepare_index_params()
index_params.add_index(
    field_name="sparse_vector",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="IP"
)
index_params.add_index(
    field_name="dense_vector",
    index_type="HNSW",
    metric_type="COSINE",
    params={"M": M, "efConstruction": efConstruction}
)

# Create the collection.
mc.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema,
    index_params=index_params,
    description="readthedocs_zilliz_langchain",
    consistency_level="Eventually",
)

print(f"Successfully created collection {COLLECTION_NAME}")

Successfully dropped collection MilvusDocs
Successfully created collection MilvusDocs


## Chunking

在嵌入之前，需要先确定您的分块策略、分块大小和分块重叠度。本节采用以下设置：

- **策略** = 简单的固定分块长度
- **分块大小** = 使用嵌入模型参数 MAX_SEQ_LENGTH
- **重叠度** = 一般建议 10%-15%
- **函数** = Langchain 的`RecursiveCharacterTextSplitter`，用于递归拆分长评论

### STEP 4. PREPARE DATA: CHUNK AND EMBED

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

# save each doc.page_content as a local file under OUTPUT_DIR
OUTPUT="./output"
os.makedirs(OUTPUT,exist_ok=True)

# Define chunk size 512 and overlap 10% chunk_size
chunk_size=512
chunk_overlap=np.round(chunk_size*0.10,0)
print(f"chunk_size: {chunk_size}, chunk_overlap: {chunk_overlap}")

# Create an instance of RecursiveCharacterTextSplitter
child_splitter=RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len, # using built-in Python len function
)

# 直接对 DirectoryLoader 加载的纯文本(page_content)分块,不再读取/解析原始 HTML。
# h1/h2 从文本尾部的 "On this page" 目录块提取(纯文本即可,不依赖 HTML 结构):
#   - h1 = 目录块第一行(文章标题)
#   - h2 = 目录块第二行(首个节标题,尽力而为;没有就留空)
start_time=time.time()
for doc in docs:
    text=doc.page_content
    tail=text.split("On this page",1)[-1]
    toc_lines=[ln.strip() for ln in tail.splitlines() if ln.strip()]
    h1=toc_lines[0][:100] if toc_lines else ""
    h2=toc_lines[1] if len(toc_lines)>1 else ""
    doc.metadata.update({"h1":h1,"h2":h2})

    # Set filename to first 50 characters of the source filename.
    filename=Path(doc.metadata['source']).stem[:50]
    filepath=os.path.join(OUTPUT,f"{filename}.html")
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(doc.page_content.replace("\n"," "))

# Split the documents into recursive chunks
chunks=child_splitter.split_documents(docs)

end_time=time.time()
print(f"chunking time: {end_time-start_time}")
print(f"docs: {len(docs)}, split into chunks: {len(chunks)}, type: list of {type(chunks[0])}")

# Inspect a chunk
print()
print("Looking at a sample chunk...")
print(chunks[0].page_content[:100])
print(chunks[0].metadata)

chunk_size: 512, chunk_overlap: 51.0
chunking time: 0.027543306350708008
docs: 22, split into chunks: 742, type: list of <class 'langchain_core.documents.base.Document'>

Looking at a sample chunk...
milvus-logo

Docs

Blog

Community

Stars0

Home

v2.4.x

About Milvus

Get Started

Concepts

Archi
{'source': 'rtdocs_new\\architecture_overview.html', 'h1': 'Milvus Architecture Overview', 'h2': "What's next"}


In [9]:
# Clean up the metadata urls
for doc in chunks:
    new_url=doc.metadata['source']
    new_url=new_url.replace("rtdocs_new","https://milvus.io/docs")
    new_url=new_url.replace(".html",".md")
    doc.metadata.update({"source":new_url})

print(chunks[0].page_content[:100])
print(chunks[0].metadata)

milvus-logo

Docs

Blog

Community

Stars0

Home

v2.4.x

About Milvus

Get Started

Concepts

Archi
{'source': 'https://milvus.io/docs\\architecture_overview.md', 'h1': 'Milvus Architecture Overview', 'h2': "What's next"}


使用内置的 Milvus BGE M3 嵌入函数。输出将包含两个向量：

- `embeddings['dense'][i]` 是一个 numpy 数组列表，每个元素对应一个分块。Milvus 支持多个密集嵌入向量，因此 i 表示第 i 个密集嵌入向量。

- `embeddings['sparse'][:, [i]]` 是一个 scipy 稀疏矩阵，其中每一列代表一个分块。

### STEP 5. TRANSFORM CHUNKS INTO VECTORS USING EMBEDDING MODEL INFERENCE

In [10]:
# BGEM3EmbeddingFunction input is docs as a list of strings.
list_of_strings=[doc.page_content for doc in chunks if hasattr(doc,'page_content')]

# Embedding inference using the milvus build-in sparse-dense-reranking encoder
start_time=time.time()
embeddings=embedding_model(list_of_strings)
end_time=time.time()

print(f"Embedding time for {len(list_of_strings)} chunks: {end_time-start_time}")

Inference Embeddings: 100%|██████████| 47/47 [00:32<00:00,  1.43it/s]


Embedding time for 742 chunks: 37.491690158843994


## Insert data into Milvus

对于每个原始文本片段，我们将把四元组（`vector, text, source, h1, h2`）写入数据库。

**Milvus 客户端封装器仅能处理从字典列表中加载数据。**

否则，Milvus 通常支持从以下格式加载数据：

- pandas 数据框
- 字典列表

下面我们将使用 HuggingFace 提供的嵌入模型，下载其检查点，并在本地运行以作为编码器。

### STEP 6. INSERT CHUNK LIST INTO MILVUS OR ZILLIZ

In [11]:
# Create chunk_list and dict_list in a single loop
dict_list=[]
for chunk,sparse,dense in zip(chunks,embeddings["sparse"],embeddings["dense"]):
    if len(sparse.shape)==1:
        sparse=sparse.reshape(1,-1)
    chunk_dict={
        'chunk':chunk.page_content,
        'h1':chunk.metadata.get('h1',"")[:50],
        'h2':chunk.metadata.get('h2',"")[:50],
        'source':chunk.metadata.get('source',""),
        'sparse_vector':sparse,
        'dense_vector':dense,
    }
    dict_list.append(chunk_dict)

# inspect the first chunk and its metadata.
print(len(dict_list))
print(type(dict_list[0]),len(dict_list[0]))
pprint.pprint(dict_list[0])

742
<class 'dict'> 6
{'chunk': 'milvus-logo\n'
          '\n'
          'Docs\n'
          '\n'
          'Blog\n'
          '\n'
          'Community\n'
          '\n'
          'Stars0\n'
          '\n'
          'Home\n'
          '\n'
          'v2.4.x\n'
          '\n'
          'About Milvus\n'
          '\n'
          'Get Started\n'
          '\n'
          'Concepts\n'
          '\n'
          'Architecture\n'
          '\n'
          'Overview\n'
          '\n'
          'Storage/Computing\n'
          '\n'
          'Main Components\n'
          '\n'
          'Data Processing\n'
          '\n'
          'Knowhere\n'
          '\n'
          'Bitset\n'
          '\n'
          'Consistency\n'
          '\n'
          'Multi-tenancy\n'
          '\n'
          'Timestamp\n'
          '\n'
          'Similarity Metrics\n'
          '\n'
          'Time Synchronization\n'
          '\n'
          'Vector Index\n'
          '\n'
          'Scalar Index\n'
          '\n'
        

In [12]:
# Insert data into the Milvus collection.
print("Start inserting entities")
start_time=time.time()
insert_result=mc.insert(
    collection_name=COLLECTION_NAME,
    data=dict_list,
    progress_bar=True,
)
elaped_time=time.time()-start_time
print(f"Milvus insert time for {len(dict_list)}: {elaped_time}")

Start inserting entities
Milvus insert time for 742: 0.22085976600646973


## Aside - example Milvus collection API calls

https://milvus.io/docs/manage-collections.md#View-Collections

以下是一些用于检查集合的常见 API 调用。

- `.num_entities`，刷新数据并执行行数统计。
- `.describe_collection()`，提供关于模式、索引和集合的详细信息。
- `.query()`，返回集合中选定的数据。

### Example Milvus Collection utility API calls.

In [13]:
mc.flush(collection_name=COLLECTION_NAME)
# Count rows, incurs a call to .flush() first.
start_time=time.time()
stats=mc.get_collection_stats(collection_name=COLLECTION_NAME)
row_count=stats["row_count"]
end_time=time.time()
print(f"Count time for {row_count} rows: {end_time-start_time}")
print()

# View collection info, incurs a call to .flush() first.
start_time=time.time()
collection_info=mc.describe_collection(collection_name=COLLECTION_NAME)
end_time=time.time()
pprint.pprint(collection_info)
print(f"time: {end_time-start_time}")
print()

# Count rows without incurring call to .flush().
start_time=time.time()
res=mc.query(
    collection_name=COLLECTION_NAME,
    filter="",
    output_fields=["count(*)"]
)
end_time=time.time()
pprint.pprint(res)
print(f"time: {end_time-start_time}")
print()

# 查询具体数据
OUTPUT_FIELDS=["id","h1","h2","source","chunk"]
res=mc.query(
    collection_name=COLLECTION_NAME,
    filter="id<468477547223198341",
    output_fields=OUTPUT_FIELDS
)
pprint.pprint(res)

Count time for 742 rows: 0.0031752586364746094

{'aliases': [],
 'auto_id': True,
 'collection_id': 468479890856609616,
 'collection_name': 'MilvusDocs',
 'consistency_level': 3,
 'consistency_level_name': 'Eventually',
 'created_timestamp': 468479932714450961,
 'description': '',
 'enable_dynamic_field': True,
 'enable_namespace': False,
 'fields': [{'auto_id': True,
             'description': '',
             'field_id': 100,
             'is_primary': True,
             'name': 'id',
             'params': {},
             'type': <DataType.INT64: 5>},
            {'description': '',
             'field_id': 101,
             'name': 'sparse_vector',
             'params': {},
             'type': <DataType.SPARSE_FLOAT_VECTOR: 104>},
            {'description': '',
             'field_id': 102,
             'name': 'dense_vector',
             'params': {'dim': 1024},
             'type': <DataType.FLOAT_VECTOR: 101>},
            {'description': '',
             'field_id': 103,


# Ask a question about your data

在本演示笔记本中，到目前为止：

1. 您的自定义数据已被映射到向量嵌入空间中。
2. 这些向量嵌入已保存至向量数据库中。
接下来，您可以针对您的自定义数据提出问题！

💡 在LLM词汇中：
> **查询**是用户提问的统称。
> 一个查询可以包含多个独立问题，最多可达1000个不同的问题！

> **问题**通常指单个用户提问。
> 在下面的例子中，用户的提问是：“Milvus Client中的AUTOINDEX是什么？”

> **语义搜索** = 针对整个知识库进行快速搜索，找出与用户查询最接近的TOP_K文档片段。

💡 为了保持一致性，所有嵌入数据和查询都应使用相同的模型。

In [14]:
# Define a sample question about your data
QUESTION1 = "What do the parameters for HNSW mean?"
QUESTION2 = "What are good default values for HNSW parameters with 25K vectors dim 1024?"
QUESTION3 = "What does nlist vs nprobe mean in ivf_flat?"
QUESTION4 = "What is the default AUTOINDEX index and vector field distance metric in Milvus?"

# In case you want to task all the questions at once
QUERY=[QUESTION1,QUESTION2,QUESTION3,QUESTION4]

# Inspect the length of one question
QUERY_LENGTH=len(QUESTION2)
print(f"Question2 length: {QUERY_LENGTH}")

Question2 length: 75


In [15]:
# SELECT A PARTICULAR QUESTION TO ASK.
# SAMPLE_QUESTION = QUESTION1

## Execute a vector search

使用 PyMilvus API 进行 Milvus 搜索。

💡 向量搜索本质上是“语义”搜索。例如，如果你搜索“leaky faucet漏水的水龙头”：

> **传统关键词搜索**——无论是“leaky”还是“faucet”，或者两者都必须与文本匹配，才能返回网页或文档链接。

> **语义搜索**——结果还会包含“drippy”、“taps”等词，因为这些词虽然不同，但含义相同。

### STEP 7. RETRIEVE ANSWERS FROM YOUR DOCUMENTS STORED IN MILVUS OR ZILLIZ.

In [16]:
from pymilvus import AnnSearchRequest, RRFRanker

# Load the index into memory for search.
mc.load_collection(collection_name=COLLECTION_NAME)

# Embed the questions using the same encoder
query_embeddings=embedding_model(QUERY)
TOP_K=2

# Return top k results with HNSW index
SEARCH_PARAMS={
    "ef":efConstruction
}

# Prepare the search requests for both vector fields
# QUERY 是列表(4 个问题),embedding_model(QUERY) 返回 nq=4 的向量,
# 每个问题的向量都要保留,不能只取第一行!

# --- 稀疏向量: (nq, 250002) 稀疏矩阵 -> list[dict],每个问题一个 dict ---
sparse_vec=query_embeddings["sparse"]
if hasattr(sparse_vec, 'tocoo'):
    # 已经是 scipy.sparse 矩阵
    coo = sparse_vec.tocoo()
    nq = coo.shape[0]
    sparse_data = []
    for q in range(nq):
        d = {}
        for i in range(coo.nnz):
            if coo.row[i] == q:
                d[int(coo.col[i])] = float(coo.data[i])
        sparse_data.append(d)
else:
    # 兜底:假定已是 list[dict] 或可迭代对象
    sparse_data = [dict(x) for x in sparse_vec]

sparse_search_params={"metric_type":"IP"}
sparse_req=AnnSearchRequest(
    sparse_data,
    "sparse_vector",
    sparse_search_params,
    limit=TOP_K,
)

# --- 稠密向量: (nq, 1024) 全保留 ---
dense_vec = np.asarray(query_embeddings["dense"], dtype=np.float32)
if dense_vec.ndim == 1:
    dense_vec = dense_vec.reshape(1, -1)

dense_search_params={"metric_type":"COSINE",**SEARCH_PARAMS}
dense_req=AnnSearchRequest(
    dense_vec,
    "dense_vector",
    dense_search_params,
    limit=TOP_K,
)

# Define output fields to return.
OUTPUT_FIELDS=["id", "h1", "h2", "source", "chunk"]

# 运行混合搜索
start_time=time.time()
# Use the reranker.
res=mc.hybrid_search(
    collection_name=COLLECTION_NAME,
    reqs=[sparse_req,dense_req],    # 请求列表
    ranker=RRFRanker(),             # 或No reranking: WeightedRanker(0.5, 0.5)
    limit=TOP_K,
    output_fields=OUTPUT_FIELDS
)
elapsed_time=time.time()-start_time
print(f"Milvus Client search time for {len(dict_list)} vectors: {elapsed_time} seconds")

# Inspect search results
print(f"type: {type(res[0])}, count: {len(res[0])}, num_queries: {len(res)}")

Milvus Client search time for 742 vectors: 0.01049041748046875 seconds
type: <class 'pymilvus.client.search_result.HybridHits'>, count: 2, num_queries: 4


## Assemble and inspect the search result

搜索结果存储在变量 `results[0]` 中，包含 top_k-count 个类型为 `pymilvus.client.abstract.Hits` 的对象。

In [17]:
# Assemble retrieved context and context metadata.
METADATA_FIELDS=[f for f in OUTPUT_FIELDS if f !='chunk']

# Assemble retrieved ids, distances, contexts, sources, and metadata.
all_ids = []
all_distances = []
all_contexts = []
all_sources = []
all_metas = []

# 遍历每个查询的结果
for query_idx, hits in enumerate(res):   # hits 是 Hits 对象，包含 TOP_K 个 Hit
    print(f"======= Query {query_idx + 1} =======")
    for hit in hits:   # 每个 Hit 对应一个匹配文档
        print(f"Retrieved result #{hit.id}")
        all_ids.append(hit.id)

        # 得分（混合搜索返回 score）
        print(f"score = {hit.score}")
        all_distances.append(hit.score)

        # 上下文内容
        chunk_text = hit.entity.get('chunk', '')
        all_contexts.append(chunk_text)
        # print(f"Context: {chunk_text[:150]}")

        # 提取元数据字段
        meta_dict = {}
        for field in METADATA_FIELDS:
            if field != 'id':   # id 已单独提取
                value = hit.entity.get(field, None)
                if value is not None:
                    meta_dict[field] = value
                    if field == 'source':
                        all_sources.append(value)
        all_metas.append(meta_dict)
        print()

# Keep results in a list of tuples.
formatted_results=list(zip(all_ids, all_distances, all_contexts, all_sources, all_metas))
pprint.pprint(formatted_results)

======= Query 1 =======
Retrieved result #468479890856610138
score = 0.032786883413791656

Retrieved result #468479890856610140
score = 0.032258063554763794

======= Query 2 =======
Retrieved result #468479890856610138
score = 0.032258063554763794

Retrieved result #468479890856610083
score = 0.016393441706895828

======= Query 3 =======
Retrieved result #468479890856610124
score = 0.016393441706895828

Retrieved result #468479890856610125
score = 0.016393441706895828

======= Query 4 =======
Retrieved result #468479890856610384
score = 0.032786883413791656

Retrieved result #468479890856610076
score = 0.016129031777381897

[(468479890856610138,
  0.032786883413791656,
  'HNSW (Hierarchical Navigable Small World Graph) is a graph-based indexing '
  'algorithm. It builds a multi-layer navigation structure for an image '
  'according to certain rules. In this structure, the upper layers are more '
  'sparse and the distances between nodes are farther; the lower layers are '
  'denser and

## Use an LLM to Generate a chat response to the user's question using the Retrieved Context.

如今存在许多不同的生成式大语言模型。请查看 lmsys 的排行榜。

在本笔记本中，我们将尝试以下这些大语言模型：

- Meta 最新发布的开源模型 Llama 3。
- Anthropic 提供的最便宜的付费模型 Claude3 Haiku。
- OpenAI 的标准型号 GPT-3.5-Turbo，其价格属于同级别中的标杆。

### STEP 8. LLM-GENERATED ANSWER TO THE QUESTION, GROUNDED BY RETRIEVED CONTEXT.

In [18]:
# Separate all the context together by space.
contexts_combined = ' '.join(reversed(all_contexts[0]))

# Separate all the source together by comma
source_combined=' '.join(all_sources[0])
print(f"Length long text to summarize: {len(contexts_combined)}")
# print(contexts_combined,' ',source_combined)

Length long text to summarize: 1011


## Try Llama3 with Ollama to generate a human-like chat response to the user's question

按照说明安装 ollama 并拉取模型。
https://github.com/ollama/ollama

查看ollama支持的模型详情。
https://ollama.com/library/llama3

该页面说明，`ollama run llama3` 默认会拉取最新的“instruct”模型，该模型已针对聊天/对话场景进行了微调。

另一种类型的 llama3 模型是“预训练”的基础模型。
示例：ollama run llama3:text ollama run llama3:70b-text

`gguf` **格式**表示模型在 CPU 上运行。gg 是“Georgi Gerganov”的缩写，他是 C 库模型格式 ggml 的创建者，该格式最近已被修改为 gguf。

**量化**（可理解为向量压缩）可以提高吞吐量，但会以降低精度为代价。感兴趣的朋友可参考以下链接了解量化方法的详细含义：
https://huggingface.co/TheBloke/Llama-2-13B-chat-GGML/tree/main。

以下是主要的量化类型：

- **q4_0**：原始量化方式，4 位。
- **q4_k_m**：将 attention.wv 和 feed_forward.w2 矩阵中一半使用 Q6_K，其余使用 Q4_K。
- **q5_0**：精度更高，资源消耗更大，推理速度较慢。
- **q5_k_m**：将 attention.wv 和 feed_forward.w2 矩阵中一半使用 Q6_K，其余使用 Q5_K。
- **q6_k**：所有张量均使用 Q8_K。
- **q8_0**：与 float16 几乎无法区分，资源消耗大且推理速度慢，不建议大多数用户使用。

In [21]:
import ollama

# Verify details which model you are running
ollama_llama3=ollama.list()['models'][0]

# Print the mode detials
keys=['format','parameter_size','quantization_level']
print(f"Model: {ollama_llama3['model']}",end=",")
for key in keys:
    print(f"{str.upper(key)}:{ollama_llama3['details'].get(key,'Key not found in dictionary')}",end=",")
print(end="\n\n")

Model: llama3:latest,FORMAT:gguf,PARAMETER_SIZE:8.0B,QUANTIZATION_LEVEL:Q4_0,



In [22]:
SYSTEM_PROMPT = f"""Given the provided context, your task is to
understand the content and accurately answer the question based
on the information available in the context.
Provide a complete, clear, concise, relevant response in fewer
than 3 sentences and cite the unique sources.
Sources: {source_combined}
Context: {contexts_combined}
"""
'''
根据提供的上下文，你的任务是
理解内容，并基于上下文中提供的信息准确回答问题。
请用不超过3句话提供完整、清晰、简洁且相关的回答，并注明来源。
'''

'\n根据提供的上下文，你的任务是\n理解内容，并基于上下文中提供的信息准确回答问题。\n请用不超过3句话提供完整、清晰、简洁且相关的回答，并注明来源。\n'

In [23]:
# Send the question to llama 3 chat.
response = ollama.chat(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT,},
        {"role": "user", "content": f"question: {QUERY[0]}",}
    ],
    model='llama3',
)
pprint.pprint(response['message']['content'].replace('\n', ' '))

('Based on the provided context, the parameters for HNSW (Hierarchical '
 'Navigable Small World) are not explicitly mentioned. However, according to '
 'the source you provided (http://milvus.io/docs/index.md), HNSW is a type of '
 'indexing algorithm that takes the following parameters:  * `M`: The number '
 'of hash tables used for indexing. * `ef`: The number of nearest neighbors to '
 'return. * `metric`: The distance metric used (e.g., L2, cosine, etc.). * '
 '`index_type`: The type of index to build (e.g., IVF, HNSW, etc.).  These '
 'parameters control the trade-off between search efficiency and recall. A '
 'detailed explanation of these parameters can be found in the Milvus '
 'documentation.  Please note that the context provided does not explicitly '
 'mention these parameters, but it is assumed that they are relevant to the '
 'discussion of HNSW.')


## Use Anthropic to generate a human-like chat response to the user's question

我们已使用开源大语言模型（LLM）在自有数据上免费练习了检索任务。
现在让我们尝试付费的 Claude3。模型列表：
- Opus — 最昂贵
- Sonnet
- Haiku — 最便宜！

提示工程教程

- 交互式
- 静态

In [24]:
SYSTEM_PROMPT = f"""Use the Context below to answer the user's question.
Be clear, factual, complete, concise.
If the answer is not in the Context, say "I don't know".
Otherwise answer with fewer than 3 sentences and cite the grounding sources.
Context: {contexts_combined}
Sources: {source_combined}

Answer with 2 parts: the answer and the source citations.
Answer: The answer to the question.
Sources: grounding source only unique urls
"""
'''
请根据以下上下文回答用户的问题。
回答要清晰、客观、完整、简洁。
如果答案不在上下文中，请说“我不知道”。
否则，用不超过三句话回答，并注明来源。
上下文：{contexts_combined}
来源：{source_combined}

回答分为两部分：答案和来源引用。
答案：问题的答案。
来源：仅列出唯一URL的来源
'''

'\n请根据以下上下文回答用户的问题。  \n回答要清晰、客观、完整、简洁。  \n如果答案不在上下文中，请说“我不知道”。  \n否则，用不超过三句话回答，并注明来源。  \n上下文：{contexts_combined}  \n来源：{source_combined}  \n\n回答分为两部分：答案和来源引用。  \n答案：问题的答案。  \n来源：仅列出唯一URL的来源\n'

In [ ]:
# import anthropic
#
# ANTHROPIC_API_KEY=os.environ.get("ANTHROPIC_API_KEY")
#
# # # Model names
# # claude-3-opus-20240229
# # claude-3-sonnet-20240229
# # claude-3-haiku-20240307
# CLAUDE_MODEL = "claude-3-haiku-20240307"
# print(f"Model: {CLAUDE_MODEL}")
# print()
#
# client = anthropic.Anthropic(
#     # defaults to os.environ.get("ANTHROPIC_API_KEY")
#     api_key=ANTHROPIC_API_KEY,
# )
#
# # Print the question and answer along with grounding sources and citations.
# print(f"Question: {SAMPLE_QUESTION}")

# # CAREFUL!! THIS COSTS MONEY!!
# message = client.messages.create(
#     model=CLAUDE_MODEL,
#     max_tokens=1000,
#     temperature=0.0,
#     system=SYSTEM_PROMPT,
#     messages=[
#         {"role": "user", "content": SAMPLE_QUESTION}
#     ]
# )
# print("Answer:")
# pprint.pprint(message.content[0].text.replace('\n', ' '))

### Try OpenAI to generate a human-like chat response to the user's question

我们已使用开源大语言模型（LLM）在自有数据上免费进行了检索练习。
现在，让我们调用付费的 OpenAI GPT。

💡 注意：对于需要始终基于事实的应用场景，请使用极低的温度值；而更具创意的任务则可适当提高温度值以获得更好的效果。

In [25]:
SYSTEM_PROMPT = f"""Use the Context below to answer the user's question.
Be clear, factual, complete, concise.
If the answer is not in the Context, say "I don't know".
Otherwise answer with fewer than 4 sentences and cite the grounding sources.
Context: {contexts_combined}
Answer: The answer to the question.
Grounding sources: {source_combined}
"""
'''
请根据以下上下文回答用户的问题。
回答要清晰、客观、完整、简洁。
如果答案不在上下文中，请说“我不知道”。
否则，用不超过4句话回答，并注明来源。
上下文：{contexts_combined}
回答：问题的答案。
来源：{source_combined}
'''

'\n请根据以下上下文回答用户的问题。\n回答要清晰、客观、完整、简洁。\n如果答案不在上下文中，请说“我不知道”。\n否则，用不超过4句话回答，并注明来源。\n上下文：{contexts_combined}\n回答：问题的答案。\n来源：{source_combined}\n'

In [ ]:
# # CAREFUL!! THIS COSTS MONEY!!
# import openai, pprint
# from openai import OpenAI
#
# # Define the generation llm model to use.
# # https://openai.com/blog/new-embedding-models-and-api-updates
# # Customers using the pinned gpt-3.5-turbo model alias will be automatically upgraded to gpt-3.5-turbo-0125 two weeks after this model launches.
# LLM_NAME = "gpt-3.5-turbo"
# TEMPERATURE = 0.1
# RANDOM_SEED = 415
#
# # See how to save api key in env variable.
# # https://help.openai.com/en/articles/5112595-best-practices-for-api-key-safety
# openai_client = OpenAI(
#     # This is the default and can be omitted
#     api_key=os.environ.get("OPENAI_API_KEY"),
# )
#
# # Generate response using the OpenAI API.
# response = openai_client.chat.completions.create(
#     messages=[
#         {"role": "system", "content": SYSTEM_PROMPT,},
#         {"role": "user", "content": f"question: {SAMPLE_QUESTION}",}
#     ],
#     model=LLM_NAME,
#     temperature=TEMPERATURE,
#     seed=RANDOM_SEED,
#     frequency_penalty=2,
# )
#
# # Print the question and answer along with grounding sources and citations.
# print(f"Question: {SAMPLE_QUESTION}")
#
# # Print all answers in the response.
# for i, choice in enumerate(response.choices, 1):
#     pprint.pprint(f"Answer: {choice.message.content}")
#     print("\n")
#
# # Question1: What do the parameters for HNSW mean?
# # Answer:  Looks perfect!
# # Best answer:  M: maximum degree of nodes in a layer of the graph.
# # efConstruction: number of nearest neighbors to consider when connecting nodes in the graph.
# # ef: number of nearest neighbors to consider when searching for similar vectors.
#
# # Question2: What are good default values for HNSW parameters with 25K vectors dim 1024?
# # Answer: M=16, efConstruction=500, and ef=64
# # Best answer:  M=16, efConstruction=32, ef=32
#
# # Question3: what is the default distance metric used in AUTOINDEX in Milvus?
# # Answer: L2
# # Trick answer:  IP inner product, not yet updated in documentation still says L2.

In [26]:
mc.drop_collection(COLLECTION_NAME)